# Colab Keyframe Extraction — AIC HCMC 2025 (K01–K20)

Tải `Videos_KXX.zip` từ `aic-data.ledo.io.vn`, dùng shots JSON (TransNetV2 đã
upload lên Drive) để extract keyframes **chất lượng gốc 1920×1080 lossless** bằng
ffmpeg (CPU), rồi đẩy videos + keyframes + zip lên Drive.

**Không cần GPU/TensorFlow.** Shot detection đã chạy xong trên local.

**Idempotent & tự phục hồi:** mọi bước dùng `aria2c -c`, `rclone --size-only`,
tmp→rename. Drive là nguồn chân lý — session bị kill thì `Run all` lại là nối tiếp.

**Cách dùng:** sửa `GROUPS` (3 nhóm/session, Colab free CPU ~2 vCPU / 13GB RAM /
~78GB đĩa / ~12h) rồi `Runtime → Run all`. Tắt "Disconnect after 90 min".

**Lưu ý:** notebook chứa refresh_token Drive cá nhân — chỉ lưu trong Drive riêng.


In [ ]:
# ===== CONFIG — chỉnh ở đây =====
GROUPS = ["K01", "K02", "K03"]   # 3 nhóm/session (Colab free CPU: ~2-3h/nhóm)

BASE_URL  = "https://aic-data.ledo.io.vn"
DEST      = "AIC_HCMC/2026/2025_Dataset"        # == /content/drive/MyDrive/AIC_HCMC/2026/2025_Dataset
SHOTS_TAR = "AIC_HCMC/2026/2025_Dataset/shots.tar.gz"

POSITIONS        = [0.15, 0.5, 0.85]   # 3 keyframe/shot
ARIA_CONNECTIONS = 16
FFMPEG_WORKERS   = 3                   # Colab free ~2 vCPU
RCLONE_TRANSFERS = 8
RCLONE_CHECKERS  = 16
# ================================


In [ ]:
import os, subprocess, urllib.request
from pathlib import Path

RC = "/usr/local/bin/rclone"

# --- aria2c ---
if subprocess.run(["which", "aria2c"], capture_output=True).returncode != 0:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "aria2"], check=True)

# --- rclone ---
if not os.path.exists(RC):
    url = "https://downloads.rclone.org/v1.75.0/rclone-v1.75.0-linux-amd64.zip"
    urllib.request.urlretrieve(url, "/tmp/rclone.zip")
    subprocess.run(["unzip", "-o", "-j", "/tmp/rclone.zip",
                    "rclone-*-linux-amd64/rclone", "-d", "/usr/local/bin"], check=True)
    subprocess.run(["chmod", "+x", RC], check=True)

# --- rclone config (upload at runtime; never embed credentials)
from google.colab import files
cfg = Path(os.path.expanduser("~/.config/rclone/rclone.conf"))
cfg.parent.mkdir(parents=True, exist_ok=True)
if not cfg.exists():
    uploaded = files.upload()
    if "rclone.conf" not in uploaded:
        raise RuntimeError("Upload a file named rclone.conf")
    cfg.write_bytes(uploaded["rclone.conf"])

subprocess.run([RC, "version"], check=True)
print("aria2 + rclone ready")

In [ ]:
import subprocess
from pathlib import Path

RC = "/usr/local/bin/rclone"
SHOTS = Path("/content/shots")

if not SHOTS.exists():
    subprocess.run([RC, "copy", f"gdrive:{SHOTS_TAR}", "/content/"], check=True)
    subprocess.run(["tar", "-xzf", "/content/shots.tar.gz", "-C", "/content/"], check=True)
    Path("/content/shots.tar.gz").unlink(missing_ok=True)

n = len(list(SHOTS.rglob("*.json")))
print(f"shots ready: {n} JSON files")


In [ ]:
import json, subprocess, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

RC = "/usr/local/bin/rclone"
WORK = Path("/content/work")
VIDEOS = WORK / "videos"
KEYFRAMES = WORK / "keyframes"
WORK.mkdir(parents=True, exist_ok=True)

PNG_SIG = b"\x89PNG\r\n\x1a\n"


def rclone_copy(src, dst):
    subprocess.run([RC, "copy", "--size-only",
                    "--transfers", str(RCLONE_TRANSFERS),
                    "--checkers", str(RCLONE_CHECKERS), src, dst], check=True)


def remote_listing(remote_dir):
    r = subprocess.run([RC, "lsf", remote_dir], capture_output=True, text=True)
    return set(r.stdout.split()) if r.returncode == 0 else set()


def remote_dir_exists(remote_dir):
    return subprocess.run([RC, "lsd", remote_dir],
                          capture_output=True).returncode == 0


def compute_keyframes(shots, positions=POSITIONS):
    indices = []
    for start, end in shots:
        for pos in positions:
            idx = int(start + (end - start) * pos)
            indices.append(max(start, min(end, idx)))
    return indices


def is_valid_png(p):
    try:
        if p.stat().st_size == 0:
            return False
        with open(p, "rb") as fh:
            return fh.read(8) == PNG_SIG
    except OSError:
        return False


def _extract_one(args):
    video, timestamp, out_path = args
    tmp = out_path.with_suffix(".tmp.png")
    try:
        subprocess.run([
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-ss", str(timestamp), "-i", str(video),
            "-frames:v", "1", "-c:v", "png", "-pix_fmt", "rgb24", str(tmp),
        ], check=True, capture_output=True, timeout=60)
        tmp.replace(out_path)
        return None
    except Exception as e:
        tmp.unlink(missing_ok=True)
        return e


def extract_frames(video, frame_indices, out_dir, fps):
    out_dir.mkdir(parents=True, exist_ok=True)
    tasks = []
    for fidx in frame_indices:
        op = out_dir / f"{fidx:06d}.png"
        if not is_valid_png(op):
            tasks.append((str(video), fidx / fps, op))
    if not tasks:
        return 0
    errs = 0
    with ThreadPoolExecutor(max_workers=FFMPEG_WORKERS) as pool:
        for e in pool.map(_extract_one, tasks):
            if e is not None:
                errs += 1
    return errs


def extract_video_zip(group):
    zpath = WORK / f"Videos_{group}.zip"
    target = VIDEOS / group
    marker = WORK / f".{group}.extracted"
    if not marker.exists():
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zpath) as zf:
            names = [m for m in zf.namelist() if not m.endswith("/")]
            tops = {m.split("/", 1)[0] for m in names}
            strip = len(tops) == 1 and all("/" in m for m in names)
            for m in names:
                rel = m.split("/", 1)[1] if strip else m
                dest = target / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                tmp = dest.with_suffix(".part")
                with zf.open(m) as src, open(tmp, "wb") as dst:
                    dst.write(src.read())
                tmp.replace(dest)
        marker.touch()
    return sorted(target.rglob("*.mp4"))


def zip_group(group):
    arc = WORK / f"{group}.zip"
    if arc.exists():
        return arc
    tmp = WORK / f"{group}.zip.tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:
        for p in sorted((KEYFRAMES / group).rglob("*.png")):
            z.write(p, p.relative_to(KEYFRAMES))
    tmp.replace(arc)
    return arc


In [ ]:
import shutil, subprocess, json
from pathlib import Path
from tqdm.auto import tqdm

SHOTS = Path("/content/shots")

done = remote_listing(f"gdrive:{DEST}/zipped")

for group in GROUPS:
    print(f"\n===== {group} =====", flush=True)

    if f"{group}.zip" in done:
        print(f"  {group}.zip already on Drive — skip", flush=True)
        continue

    # 1. download video zip (resumable)
    zpath = WORK / f"Videos_{group}.zip"
    if not zpath.exists() or zpath.stat().st_size == 0:
        subprocess.run([
            "aria2c", "-d", str(WORK), "-o", f"Videos_{group}.zip",
            "-x", str(ARIA_CONNECTIONS), "-s", str(ARIA_CONNECTIONS),
            "-c", "--file-allocation=none", "--max-tries=5", "--retry-wait=10",
            "--console-log-level=warn", "--summary-interval=30",
            f"{BASE_URL}/Videos_{group}.zip",
        ], check=True)

    # 2. unzip videos, then delete zip (save disk)
    videos = extract_video_zip(group)
    zpath.unlink(missing_ok=True)
    print(f"  {len(videos)} videos", flush=True)

    # 3. pull existing keyframes from Drive (resume, skip re-extract)
    if remote_dir_exists(f"gdrive:{DEST}/keyframes/{group}"):
        rclone_copy(f"gdrive:{DEST}/keyframes/{group}", str(KEYFRAMES / group))

    # 4. extract missing keyframes (ffmpeg, 1080p lossless)
    shots_dir = SHOTS / group
    for v in tqdm(videos, desc=f"{group} extract", unit="vid"):
        sf = shots_dir / f"{v.stem}.json"
        if not sf.exists():
            tqdm.write(f"  MISSING shots {sf.name} — skip")
            continue
        info = json.loads(sf.read_text())
        out_dir = KEYFRAMES / v.relative_to(VIDEOS).with_suffix("")
        extract_frames(v, compute_keyframes(info["shots"]), out_dir, info["fps"])

    # 4b. retry once (recovers transient ffmpeg failures, skips valid PNGs)
    for v in tqdm(videos, desc=f"{group} recover", unit="vid"):
        sf = shots_dir / f"{v.stem}.json"
        if sf.exists():
            info = json.loads(sf.read_text())
            extract_frames(v, compute_keyframes(info["shots"]),
                           KEYFRAMES / v.relative_to(VIDEOS).with_suffix(""), info["fps"])

    # 5. push keyframes to Drive
    rclone_copy(str(KEYFRAMES / group), f"gdrive:{DEST}/keyframes/{group}")

    # 6. zip + upload zip, then delete keyframes + zip (save disk)
    arc = zip_group(group)
    rclone_copy(str(arc), f"gdrive:{DEST}/zipped/")
    print(f"  uploaded {group}.zip ({arc.stat().st_size/1e9:.2f} GB)", flush=True)
    shutil.rmtree(KEYFRAMES / group, ignore_errors=True)
    arc.unlink(missing_ok=True)

    # 7. upload videos, then delete (save disk)
    rclone_copy(str(VIDEOS / group), f"gdrive:{DEST}/videos/{group}")
    shutil.rmtree(VIDEOS / group, ignore_errors=True)

print("\nALL DONE")


In [ ]:
import subprocess

RC = "/usr/local/bin/rclone"
for sub in ["zipped", "videos", "keyframes"]:
    r = subprocess.run([RC, "lsf", f"gdrive:{DEST}/{sub}"], capture_output=True, text=True)
    entries = r.stdout.split()
    print(f"gdrive:{DEST}/{sub}/  -> {len(entries)} entries")
    print("  " + (" ".join(entries) if entries else "(empty)"))
